# NB02 — Dataset Manifest & Provenance

Scans the read-only WSI root, opens each slide to extract dimensions, mpp, vendor, and objective power, computes quick fingerprints for duplicate detection, and writes the manifest as both parquet and CSV. Also produces summary diagnostic figures (size distribution, width/height log distribution, slides per cancer code, mpp availability).

In [ ]:
import os, sys, json, time, hashlib, datetime
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import openslide

WORKSPACE = Path(os.environ.get('WORKSPACE', './workspace'))
WSI_ROOT  = Path(os.environ.get('WSI_ROOT',  './data/wsi'))
SUBDIRS = {
    'compute':   WORKSPACE / 'compute',
    'logs':      WORKSPACE / 'logs',
    'figures':   WORKSPACE / 'figures',
    'manifests': WORKSPACE / 'manifests',
    'hashes':    WORKSPACE / 'hashes',
}
for p in SUBDIRS.values():
    p.mkdir(parents=True, exist_ok=True)

MANIFEST_OUT  = SUBDIRS['manifests'] / 'manifest_tcga.parquet'
MANIFEST_CSV  = SUBDIRS['manifests'] / 'manifest_tcga.csv'
FAILED_CSV    = SUBDIRS['manifests'] / 'failed_slides.csv'
HASH_INDEX    = SUBDIRS['hashes']    / 'hash_index_tcga.csv'
CHECKSUM_MODE = 'sha1_quick'
MAX_WORKERS   = min(12, (os.cpu_count() or 8))

def now_iso():
    return datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')

def file_times(p: Path):
    st = p.stat()
    created = datetime.datetime.fromtimestamp(getattr(st, 'st_ctime', st.st_mtime)).strftime('%Y-%m-%d %H:%M:%S')
    modified = datetime.datetime.fromtimestamp(st.st_mtime).strftime('%Y-%m-%d %H:%M:%S')
    return created, modified

def quick_fingerprint(path: Path, mode='sha1_quick', chunk=8*1024*1024):
    size = path.stat().st_size
    if mode == 'size_only':
        return f'SIZE:{size}', None
    if mode == 'sha1_quick':
        h = hashlib.sha1()
        with path.open('rb') as f:
            h.update(f.read(chunk))
            if size > chunk:
                f.seek(max(size - chunk, 0))
                h.update(f.read(chunk))
        h.update(str(size).encode('utf-8'))
        return f'QSHA1:{h.hexdigest()}', None
    if mode == 'sha1_full':
        h = hashlib.sha1()
        with path.open('rb') as f:
            while True:
                b = f.read(1024*1024)
                if not b: break
                h.update(b)
        return f'SHA1:{h.hexdigest()}', h.hexdigest()
    raise ValueError(f'unknown CHECKSUM_MODE: {mode}')

def list_wsi_files(root: Path):
    exts = ('.svs', '.tif', '.tiff', '.ndpi', '.mrxs', '.scn')
    out = []
    for ext in exts:
        out.extend(root.rglob(f'*{ext}'))
        out.extend(root.rglob(f'*{ext.upper()}'))
    return sorted(set(out))

def cancer_code_from_path(p: Path, root: Path):
    rel = p.relative_to(root)
    return rel.parts[0] if len(rel.parts) >= 2 else 'UNKNOWN'

def open_and_probe(path: Path):
    slide = openslide.OpenSlide(str(path))
    props = slide.properties
    width, height = slide.dimensions
    level_count = slide.level_count
    mpp_x = props.get('openslide.mpp-x') or props.get('aperio.MPP') or None
    mpp_y = props.get('openslide.mpp-y') or props.get('aperio.MPP') or None
    vendor = props.get('openslide.vendor') or 'unknown'
    obj_pow = props.get('aperio.AppMag') or props.get('openslide.objective-power') or None
    slide.close()
    return {
        'width': int(width), 'height': int(height), 'level_count': int(level_count),
        'mpp_x': float(mpp_x) if mpp_x not in (None, '') else None,
        'mpp_y': float(mpp_y) if mpp_y not in (None, '') else None,
        'vendor': str(vendor),
        'objective_power': float(obj_pow) if (obj_pow is not None and str(obj_pow).replace('.', '', 1).isdigit()) else (str(obj_pow) if obj_pow else None),
    }

start = time.time()
print(f'[{now_iso()}] scanning WSI root (read-only): {WSI_ROOT}')
slides = list_wsi_files(WSI_ROOT)
n_total = len(slides)
print(f'[INFO] found {n_total} candidate WSI files')

records = []
failures = []

def process_one(path: Path):
    rec = {
        'path': str(path), 'filename': path.name, 'slide_id': path.stem,
        'cancer_code': cancer_code_from_path(path, WSI_ROOT),
        'size_bytes': path.stat().st_size,
    }
    created, modified = file_times(path)
    rec['created_time'] = created
    rec['modified_time'] = modified
    try:
        fp, sha1_full = quick_fingerprint(path, mode=CHECKSUM_MODE)
        rec['fingerprint'] = fp
        rec['sha1_full'] = sha1_full
    except Exception:
        rec['fingerprint'] = None; rec['sha1_full'] = None
    try:
        meta = open_and_probe(path)
        rec.update(meta); rec['error'] = None
    except Exception as e:
        rec.update({'width': None, 'height': None, 'level_count': None,
                    'mpp_x': None, 'mpp_y': None, 'vendor': None, 'objective_power': None,
                    'error': f'{e.__class__.__name__}: {e}'})
    return rec

t0 = time.time()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(process_one, p): p for p in slides}
    done = 0; last_print = t0
    for fut in as_completed(futures):
        rec = fut.result()
        records.append(rec)
        if rec.get('error'):
            failures.append({'path': rec['path'], 'error': rec['error']})
        done += 1
        now = time.time()
        if now - last_print > 2 or done == n_total:
            rate = done / (now - t0 + 1e-9)
            print(f'  scanned {done}/{n_total} ({rate:.1f} files/s)')
            last_print = now

elapsed_scan = time.time() - start
print(f'[OK] scanned {n_total} slides in {elapsed_scan/60:.1f} min')

df = pd.DataFrame.from_records(records)
for c in ['size_bytes', 'width', 'height', 'level_count', 'mpp_x', 'mpp_y']:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

df.to_parquet(MANIFEST_OUT, index=False)
df.to_csv(MANIFEST_CSV, index=False, encoding='utf-8-sig')
print(f'[OK] manifest: {MANIFEST_OUT}')
print(f'[OK] manifest: {MANIFEST_CSV}')

if failures:
    pd.DataFrame(failures).to_csv(FAILED_CSV, index=False, encoding='utf-8-sig')
    print(f'[WARN] {len(failures)} slides failed to open; see {FAILED_CSV}')

df[['path', 'size_bytes', 'fingerprint']].to_csv(HASH_INDEX, index=False, encoding='utf-8-sig')
print(f'[OK] hash index: {HASH_INDEX}')

total_bytes = df['size_bytes'].sum(skipna=True)
print(f'\n  total slides: {len(df):,}')
print(f'  total size:   {total_bytes/(1024**3):.2f} GB')
by_cancer = df['cancer_code'].value_counts(dropna=False)
print('\n  slides by cancer_code (top 20):')
print(by_cancer.head(20).to_string())

missing_mpp = df[df['mpp_x'].isna() | df['mpp_y'].isna()]
print(f'\n  missing mpp entries: {len(missing_mpp)}')

fig_dir = SUBDIRS['figures']
fig_dir.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(8,5))
sizes_gb = (df['size_bytes']/(1024**3)).dropna()
plt.hist(sizes_gb.values, bins=40)
plt.xlabel('Slide size (GB)'); plt.ylabel('Count')
plt.title('WSI size distribution (TCGA)')
plt.tight_layout()
p1 = fig_dir / 'manifest_size_distribution.png'
plt.savefig(p1, dpi=200); plt.close()

plt.figure(figsize=(8,5))
wh = df[['width', 'height']].dropna()
vals = np.log10(wh.values.clip(min=1))
plt.hist(vals.flatten(), bins=40)
plt.xlabel('log10(pixels)'); plt.ylabel('Count')
plt.title('WSI width/height distribution (log10)')
plt.tight_layout()
p2 = fig_dir / 'manifest_wh_log_distribution.png'
plt.savefig(p2, dpi=200); plt.close()

plt.figure(figsize=(10,6))
top_codes = by_cancer.head(30)
plt.bar(top_codes.index.astype(str), top_codes.values)
plt.xticks(rotation=80, ha='right')
plt.ylabel('Slides')
plt.title('Slides per cancer code (top 30)')
plt.tight_layout()
p3 = fig_dir / 'manifest_counts_by_cancer.png'
plt.savefig(p3, dpi=200); plt.close()

mpp_complete = df['mpp_x'].notna() & df['mpp_y'].notna()
pct_mpp = 100.0 * mpp_complete.mean()
plt.figure(figsize=(4,4))
plt.bar(['mpp complete', 'mpp missing'], [pct_mpp, 100.0 - pct_mpp])
plt.title('mpp availability (%)')
plt.tight_layout()
p4 = fig_dir / 'manifest_mpp_availability.png'
plt.savefig(p4, dpi=200); plt.close()

compute_path = SUBDIRS['compute'] / 'compute_passport.json'
try:
    with compute_path.open('r', encoding='utf-8') as f:
        cp = json.load(f)
except Exception:
    cp = {'stages': []}
stage_entry = {
    'stage': 'manifest_tcga', 'timestamp': now_iso(),
    'inputs': {'wsi_root': str(WSI_ROOT)},
    'outputs': {
        'manifest_parquet': str(MANIFEST_OUT),
        'manifest_csv': str(MANIFEST_CSV),
        'failed_csv': str(FAILED_CSV) if failures else None,
        'hash_index_csv': str(HASH_INDEX),
        'figures': [str(p1), str(p2), str(p3), str(p4)],
    },
    'stats': {
        'n_files_found': int(n_total),
        'n_records': int(len(df)),
        'n_failures': int(len(failures)),
        'total_gb': float(total_bytes/(1024**3)),
        'elapsed_minutes': float(elapsed_scan/60.0),
        'checksum_mode': CHECKSUM_MODE,
    },
}
cp.setdefault('stages', []).append(stage_entry)
tmp = compute_path.parent / (compute_path.name + '.tmp')
with tmp.open('w', encoding='utf-8') as f:
    json.dump(cp, f, ensure_ascii=False, indent=2)
tmp.replace(compute_path)
print(f'\n[OK] compute passport updated: {compute_path}')
print('NB02 complete. Next: NB03 (QC & tissue masking).')